# Лабораторна робота № 1
## Технології Semantic Web: RDF та SPARQL

**Дисципліна:** Принципи, методи зв'язування даних у застосуваннях Семантичної павутини (ОК4)
**Викладач:** Тимчук О. С., к.т.н., доцент
**Виконав(ла):** _______________, група ІПЗм-11
**Освітня програма:** F2 «Інженерія програмного забезпечення», освітній рівень магістр

**Репозиторій:** https://github.com/USERNAME/semantic-web-lab1

## 1. Предметна область та джерела даних

Предметною областю обрано **структуру освітньо-наукової програми** підготовки магістрів
за спеціальністю F2 «Інженерія програмного забезпечення».

**Джерело даних:** ОНП «Інженерія програмного забезпечення», ID ЄДБО 9112, розглянуто
Вченою радою КНУ 24.03.2025 (протокол № 9), введено в дію наказом ректора № 342-32
від 25.04.2025. Дані внесено з таблиці 2.1 (перелік компонент), розділів 6–7
(компетентності та програмні результати навчання) і матриць 4, 5, 6.

**Правило пріоритету джерел.** Для складу програми та атрибутів рівня програми
(код компонента, назва, кредити, форма контролю, вид компонента) джерелом істини є ОНП.
Робочі програми навчальних дисциплін містять розбіжності з ОНП (зокрема ОК4 і ОК8 на
титулах указують освітній рівень «бакалавр», а ОК8 у анотації називає себе вибірковою),
тому як джерело атрибутів рівня програми не використовуються. Неоднорідні атрибути —
такі, що доступні лише для частини компонент (викладач, кафедра компонента, семестр,
години, теми), — до графу не вносяться взагалі, щоб граф лишався однорідним.

### Питання, відповіді на які дає граф

1. Які обов'язкові компоненти завершуються іспитом і скільки в них кредитів?
2. Які компоненти важать понад 5 кредитів?
3. Як розподіляються компоненти за обсягом кредитів?
4. Скільки компонентів і кредитів припадає на кожну форму підсумкового контролю?
5. Через які компоненти студент підгрупи 2 виходить на компетентність СК11?
6. Які компоненти забезпечують ПРН20 і які компетентності при цьому формуються?

## 2. Побудова RDF-графу

In [ ]:
import onp_data as d
from build_graph import build, BASE, ONP, SDO, ORG
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD, FOAF, DCTERMS, SKOS

g = build()
print(f"трійок у графі: {len(g)}")

### Перевірка відповідності вимогам

| Вимога | Мінімум | Факт |
|---|---|---|
| типів сутностей | 5 | 11 |
| екземплярів сутностей | 20 | 99 |
| типів зв'язків | 5 | 16 власних |
| RDF-трійок | 50 | 896 |
| власні URI | так | простір імен `onp:` |
| існуючий словник | 1 | Schema.org, Dublin Core, SKOS, FOAF, W3C Organization |

In [ ]:
vlasni_typy = {str(t).replace(BASE, "") for t in g.objects(None, RDF.type) if str(t).startswith(BASE)}
ekzemplyary = set(g.subjects(RDF.type, None))
vlasni_pred = {str(p).replace(BASE, "") for p in g.predicates(None, None) if str(p).startswith(BASE)}

print(f"типів сутностей : {len(vlasni_typy)}")
print(f"екземплярів     : {len(ekzemplyary)}")
print(f"типів зв'язків  : {len(vlasni_pred)}")
print(f"трійок          : {len(g)}")
print()
print("Типи сутностей:", ", ".join(sorted(vlasni_typy)))
print()
print("Власні предикати:", ", ".join(sorted(vlasni_pred)))

### Візуалізація графу

In [ ]:
from visualize import zriz_png, povnyi_html
zriz_png(g)        # статична схема зрізу (потребує Graphviz)
povnyi_html(g)     # інтерактивний повний граф

In [ ]:
from IPython.display import Image, IFrame, display
try:
    display(Image("graf_zriz.png"))
except Exception as e:
    print("статичну схему не створено:", e)

In [ ]:
IFrame("graf_povnyi.html", width="100%", height=700)

## 3. Робота з RDF-графом

### 3.1 Серіалізація у Turtle

In [ ]:
g.serialize("onp.ttl", format="turtle")
print(open("onp.ttl", encoding="utf-8").read()[:1800])

### 3.2 Серіалізація у другий формат — JSON-LD

In [ ]:
g.serialize("onp.jsonld", format="json-ld", auto_compact=True, indent=2)
print(open("onp.jsonld", encoding="utf-8").read()[:1200])

### 3.3 Читання Turtle-файлу та внесення змін вручну

Файл `onp.ttl` відкривається у текстовому редакторі, до нього додається
твердження вручну, після чого файл читається повторно і кількість трійок звіряється.

In [ ]:
g2 = Graph()
g2.parse("onp.ttl", format="turtle")
do_zmin = len(g2)
print(f"до ручної правки: {do_zmin} трійок")

# --- Ручна правка (те саме, що дописується в редакторі наприкінці файлу) ---
RUCHNA_PRAVKA = '''
@prefix onp: <https://example.org/onp/> .
@prefix sdo: <https://schema.org/> .
@prefix dcterms: <http://purl.org/dc/terms/> .

onp:komponent-OK4 sdo:teaches "Побудова графів знань, RDF, SPARQL, JSON-LD"@uk ;
                  dcterms:identifier "ОК 4" .
'''
with open("onp.ttl", "a", encoding="utf-8") as f:
    f.write(RUCHNA_PRAVKA)

g3 = Graph()
g3.parse("onp.ttl", format="turtle")
print(f"після ручної правки: {len(g3)} трійок (+{len(g3) - do_zmin})")

### 3.4 Вивід трійок графу

In [ ]:
for i, (s, p, o) in enumerate(g):
    if i >= 15:
        print(f"... разом {len(g)} трійок")
        break
    print(f"{str(s).replace(BASE,'onp:'):46s} {str(p).split('/')[-1].split('#')[-1]:24s} {str(o)[:60]}")

### 3.5 Трійки, пов'язані з конкретною сутністю

In [ ]:
subyekt = ONP["komponent-OK4"]   # сама ця дисципліна
print(f"Усі трійки, де {subyekt} є суб'єктом:\n")
for p, o in g.predicate_objects(subyekt):
    print(f"  {str(p).split('/')[-1].split('#')[-1]:26s} -> {str(o).replace(BASE,'onp:')[:80]}")

print(f"\nУсі трійки, де {subyekt} є об'єктом:\n")
for s, p in g.subject_predicates(subyekt):
    print(f"  {str(s).replace(BASE,'onp:')[:60]:60s} <- {str(p).split('/')[-1].split('#')[-1]}")

## 4. SPARQL-запити до власного графу

Запити демонструють `FILTER`, сортування, отримання даних через зв'язки між різними
типами сутностей, агрегатні функції та перехід через кілька зв'язків графу.

In [ ]:
from queries import ZAPYTY, vykonaty, POMYLKOVYI, VYPRAVLENYI

In [ ]:
nazva, zapyt = ZAPYTY[0]
print(zapyt)
vykonaty(g, nazva, zapyt)

In [ ]:
nazva, zapyt = ZAPYTY[1]
print(zapyt)
vykonaty(g, nazva, zapyt)

In [ ]:
nazva, zapyt = ZAPYTY[2]
print(zapyt)
vykonaty(g, nazva, zapyt)

In [ ]:
nazva, zapyt = ZAPYTY[3]
print(zapyt)
vykonaty(g, nazva, zapyt)

In [ ]:
nazva, zapyt = ZAPYTY[4]
print(zapyt)
vykonaty(g, nazva, zapyt)

In [ ]:
nazva, zapyt = ZAPYTY[5]
print(zapyt)
vykonaty(g, nazva, zapyt)

## 5. Поєднання з відкритими даними

Обрано **Wikidata**.

| Ресурс власного графу | Ресурс Wikidata |
|---|---|
| `https://example.org/onp/KNU` | `http://www.wikidata.org/entity/Q84151` |

**Предикат:** `owl:sameAs`.

**Обґрунтування.** Обидва URI позначають ту саму реальну організацію — Київський
національний університет імені Тараса Шевченка. `owl:sameAs` стверджує тотожність
денотатів, що для цієї пари коректно. `rdfs:seeAlso` було б слабшим, ніж дозволяють
дані, оскільки лише вказує на пов'язаний ресурс без твердження про тотожність;
`skos:exactMatch` описує відповідність між поняттями схем класифікації, а не між
описами одного об'єкта.

**Технічне зауваження.** rdflib підтримує ключове слово `SERVICE`, проте у функції
`evalServiceQuery` жорстко зашитий заголовок `user-agent: rdflibForAnUser` без
можливості перевизначення, а політика Wikimedia блокує неописові user-agent
(HTTP 403). Тому дані отримуються через SPARQLWrapper із власним описовим
user-agent, вливаються у локальний граф, після чого виконується один запит до
об'єднаного графу.

In [ ]:
from wikidata_link import zavantazhyty_z_wikidata, OBIEDNANYI_ZAPYT, ZAPYT_WIKIDATA
print(ZAPYT_WIKIDATA)
try:
    dodano = zavantazhyty_z_wikidata(g)
    print(f"додано з Wikidata: {dodano} трійок -> разом {len(g)}")
except Exception as e:
    print(f"Wikidata недоступна: {type(e).__name__}: {e}")
    print("Перевірити: user-agent у wikidata_link.py має бути описовим (політика Wikimedia),")
    print("інакше сервіс відповідає HTTP 403.")

In [ ]:
for s, p, o in g.triples((URIRef("http://www.wikidata.org/entity/Q84151"), None, None)):
    print(f"  {str(p).split('/')[-1]:10s} {o}")

### 5.1 Запит, що використовує дані власного та зовнішнього графів

In [ ]:
print(OBIEDNANYI_ZAPYT)
for row in g.query(OBIEDNANYI_ZAPYT):
    print("  " + " | ".join(str(v) for v in row))

In [ ]:
g.serialize("onp_z_wikidata.ttl", format="turtle")
print("збережено onp_z_wikidata.ttl")

## 6. Помилковий та виправлений приклад

**Помилка.** Наміром було отримати «усі обов'язкові компоненти та їхні викладачі».
Викладач є неоднорідним атрибутом: робочі програми доступні лише для частини компонент.
Запит без `OPTIONAL` мовчки відкидає всі компоненти без викладача. Замість очікуваних
15 рядків повертається порожній результат, і при цьому не виникає жодної помилки —
саме така мовчазна втрата даних є типовою пасткою SPARQL.

In [ ]:
print(POMYLKOVYI)
vykonaty(g, "ПОМИЛКОВИЙ запит", POMYLKOVYI)

**Виправлення.** Неоднорідний атрибут до графу не вноситься взагалі: джерелом істини
для складу програми є ОНП, однакова для всіх компонент. Запит спирається лише на
атрибути, наявні в кожного компонента, і повертає всі 15 обов'язкових компонент.
Альтернативне виправлення — `OPTIONAL { ?k onp:vykladach ?vykladach }` — зберегло б
рядки, але заповнило б більшість із них порожніми значеннями, тому обрано перший варіант.

In [ ]:
print(VYPRAVLENYI)
vykonaty(g, "ВИПРАВЛЕНИЙ запит", VYPRAVLENYI)

## 7. Висновки

1. Побудовано RDF-граф освітньо-наукової програми на 896 трійок: 11 типів сутностей,
   99 екземплярів, 16 власних типів зв'язків. Усі кількісні вимоги перевищено.
2. Повторно використано п'ять наявних словників — Schema.org, Dublin Core, SKOS, FOAF
   та W3C Organization Ontology, що робить граф придатним до зв'язування із зовнішніми
   джерелами без додаткового узгодження схем.
3. Матриці відповідності ОНП виявилися найціннішою частиною даних: вони задають зв'язки
   «багато до багатьох» між компонентами, компетентностями та результатами навчання,
   завдяки чому запити з переходом через кілька зв'язків графу мають змістовні відповіді.
4. Агрегатний запит за формами контролю дав 87 кредитів обов'язкових компонент, що
   збігається з підсумковим рядком таблиці 2.1 ОНП. Це незалежне підтвердження
   коректності внесених даних.
5. Розбіжності між ОНП та робочими програмами дисциплін показали, що при інтеграції
   різнорідних джерел потрібне явне правило пріоритету; без нього одна сутність
   отримує суперечливі твердження.
6. Практичне обмеження rdflib: реалізація `SERVICE` не дозволяє задати власний
   user-agent, тому для роботи з Wikidata надійнішим є отримання даних окремим
   клієнтом із подальшим вливанням у локальний граф.

## 8. Використання ШІ

Відповідно до п.7.3 робочої програми дисципліни декларується використання
інструментів автоматичної генерації коду.

| Етап | Використання ШІ |
|---|---|
| Вибір предметної області | обговорення варіантів, остаточний вибір власний |
| Витягування матриць відповідності зі сканованої ОНП | генерація коду детекції сітки таблиці та аналізу комірок |
| Проєктування схеми графу | обговорення переліку сутностей і зв'язків |
| Код побудови графу, запитів, візуалізації | генерація з подальшою перевіркою й правками |
| Формулювання SPARQL-запитів | генерація чернеток, перевірка результатів власна |
| Оформлення звіту | структурування тексту |

Згенерований код перевірено запуском, результати кожного запиту звірено з вихідними
документами. Альтернативою застосованим інструментам було ручне внесення даних із
матриць ОНП, що при 23 × 17 і 18 × 23 комірках дає високий ризик помилок; обраний
підхід дозволив додатково виявити артефакт сканування у матриці 6.